# Hybrid Quantum CNN (Quanvolutional LeNet) on MNIST

A **quanvolutional layer** (Henderson et al., 2019, made trainable here) replaces the first convolution of a LeNet-style CNN.
A 4-qubit variational circuit slides over the image as a 2×2 kernel with stride 2; each patch becomes one circuit, and the 4 Pauli-Z expectations become 4 output channels.

```
 input 1×28×28  (pixels in [0,1])
   │  2×2 patches, stride 2  → 196 patches / image
   ▼
 QUANV  4 qubits:  RY(π·p_k) encoding → [Rot(φ,θ,ω) ⊗4 → CNOT ring] × L → ⟨Z_0..Z_3⟩      ← quantum
   │  4×14×14 quantum feature maps
   ▼
 Conv 5×5 (4→16) → ReLU → MaxPool 2      16×5×5                                            ┐
 FC 400→120 → ReLU → FC 120→84 → ReLU → FC 84→10                                           ┘ classical
```

Encoding of a patch $\mathbf p=(p_0,p_1,p_2,p_3)$, then $L$ trainable layers:
$$|\psi(\mathbf p)\rangle=\prod_{l=1}^{L}\Big[\mathrm{CNOT}_{\text{ring}}\bigotimes_{k=0}^{3}\mathrm{Rot}(\phi_{lk},\theta_{lk},\omega_{lk})\Big]\bigotimes_{k=0}^{3}R_Y(\pi p_k)|0\rangle^{\otimes4},\qquad
f_k(\mathbf p)=\langle\psi|Z_k|\psi\rangle$$

The same circuit (shared weights) is applied to every patch, like a convolution kernel. It is simulated exactly with a batched, differentiable PyTorch statevector simulator: all patches of all images form one batch. The simulator is validated against PennyLane in section 5b.

**Comparison models (section 13):**
| Model | First layer |
|---|---|
| Hybrid QCNN (trainable) | trainable quanvolution, 4 qubits × L layers |
| Hybrid QCNN (random, fixed) | same circuit, weights frozen at random init (original Henderson et al. setup) |
| Classical control | Conv2d 1→4, 2×2, stride 2, + tanh (same output shape and range) |
| Classical LeNet-5 | Conv2d 1→6, 5×5 + ReLU + MaxPool (standard) |

## 1. Setup

In [ ]:
# !pip install torch torchvision matplotlib scikit-learn tqdm pennylane   # pennylane optional (validation + drawing)
import time, random, math, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.auto import tqdm

%matplotlib inline
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## 2. Configuration

In [ ]:
CFG = dict(
    # quanvolution
    q_layers      = 2,       # variational layers in the quanv circuit
    q_trainable   = True,    # False = fixed random circuit (Henderson et al. 2019)
    shots         = None,    # None = exact <Z>; int = finite-shot estimate at eval time
    # training
    epochs        = 10,
    batch_size    = 128,
    lr            = 1e-3,    # classical parameters
    q_lr          = 1e-2,    # quantum circuit parameters
    seed          = 42,
    data_root     = "./data",
    ckpt          = "best_hybrid_qcnn_mnist.pt",
    run_baselines = True,    # train comparison models in section 13
)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed(CFG["seed"])
CFG

## 3. MNIST data
Pixels kept in [0,1]: they map directly to rotation angles π·p.

In [ ]:
tf = transforms.ToTensor()
train_ds = datasets.MNIST(CFG["data_root"], train=True,  download=True, transform=tf)
test_ds  = datasets.MNIST(CFG["data_root"], train=False, download=True, transform=tf)

pin = torch.cuda.is_available()
train_dl = DataLoader(train_ds, CFG["batch_size"], shuffle=True,  num_workers=0, pin_memory=pin, drop_last=True)
test_dl  = DataLoader(test_ds,  CFG["batch_size"], shuffle=False, num_workers=0, pin_memory=pin)
print(f"train: {len(train_ds)}  test: {len(test_ds)}  batches/epoch: {len(train_dl)}")

## 4. Quantum circuit simulator (batched, differentiable)

State tensor `(N, 2**n)` with N = batch × patches; wire 0 = most significant bit (PennyLane convention).
* **Encoding**: $\bigotimes_k R_Y(\pi p_k)|0\rangle$ is a product state, so it is built directly as a Kronecker product of $(\cos\frac{\pi p_k}{2}, \sin\frac{\pi p_k}{2})$.
* **Rot**: per-wire 2×2 contraction; **CNOT ring**: a fixed basis permutation.
* **Readout**: $\langle Z_k\rangle=\sum_j|\psi_j|^2 z_k(j)$, as one matmul.

In [ ]:
def apply_1q(state, U, wire, n):
    """state (N, 2**n); U (2,2) applied to `wire`."""
    N = state.shape[0]
    s = state.view(N, 2 ** wire, 2, 2 ** (n - wire - 1))
    return torch.einsum("ij,bajc->baic", U, s).reshape(N, 2 ** n)

def rot_matrix(phi, theta, omega):
    """PennyLane Rot(phi,theta,omega) = RZ(omega) RY(theta) RZ(phi); inputs (k,) -> (k,2,2) complex."""
    c, s = torch.cos(theta / 2), torch.sin(theta / 2)
    ep = torch.exp(-0.5j * (phi + omega).to(torch.complex64))
    em = torch.exp(-0.5j * (phi - omega).to(torch.complex64))
    return torch.stack([torch.stack([ep * c, -em.conj() * s], -1),
                        torch.stack([em * s,  ep.conj() * c], -1)], -2)

def ry_product_state(angles):
    """angles (N, n) -> real amplitudes of ⊗_k RY(angle_k)|0>, shape (N, 2**n)."""
    c, s = torch.cos(angles / 2), torch.sin(angles / 2)
    state = torch.stack([c[:, 0], s[:, 0]], 1)
    for k in range(1, angles.shape[1]):
        q = torch.stack([c[:, k], s[:, k]], 1)
        state = (state.unsqueeze(2) * q.unsqueeze(1)).reshape(state.shape[0], -1)
    return state


class QuanvCircuit(nn.Module):
    """n-qubit variational kernel: RY(π p) encoding -> [Rot ⊗n -> CNOT ring] × L -> <Z_k>.
    input  patches (N, n) in [0,1]  ->  output (N, n) in [-1,1]."""

    def __init__(self, n_qubits=4, n_layers=2, trainable=True, shots=None):
        super().__init__()
        self.n, self.L, self.shots = n_qubits, n_layers, shots
        self.weights = nn.Parameter(torch.rand(n_layers, n_qubits, 3) * 2 * math.pi,
                                    requires_grad=trainable)               # (phi, theta, omega)
        n, dim = n_qubits, 2 ** n_qubits
        bits = [[(j >> (n - 1 - k)) & 1 for k in range(n)] for j in range(dim)]
        def ring(b):
            b = list(b)
            for c in range(n):
                if b[c]: b[(c + 1) % n] ^= 1
            return b
        f = torch.tensor([int("".join(map(str, ring(b))), 2) for b in bits])
        self.register_buffer("perm", torch.argsort(f))
        self.register_buffer("zsign", 1.0 - 2.0 * torch.tensor(bits, dtype=torch.float32))

    def forward(self, p):
        n = self.n
        state = ry_product_state(math.pi * p).to(torch.complex64)
        for l in range(self.L):
            U = rot_matrix(self.weights[l, :, 0], self.weights[l, :, 1], self.weights[l, :, 2])
            for w in range(n):
                state = apply_1q(state, U[w], w, n)
            state = state[:, self.perm]                                    # CNOT ring
        z = (state.real ** 2 + state.imag ** 2) @ self.zsign
        if self.shots and not self.training:                               # finite-shot emulation
            p1 = ((1 - z) / 2).clamp(0, 1)
            z = 1 - 2 * torch.distributions.Binomial(self.shots, p1).sample() / self.shots
        return z

    def extra_repr(self):
        return (f"n_qubits={self.n}, layers={self.L}, trainable={self.weights.requires_grad}, "
                f"params={self.weights.numel()}, shots={self.shots}")


class QuanvLayer(nn.Module):
    """Quantum convolution: 2x2 kernel, stride 2, shared circuit over all patches.
    (B,1,H,W) -> (B,4,H/2,W/2)."""

    def __init__(self, n_layers=2, trainable=True, shots=None, kernel=2, stride=2):
        super().__init__()
        self.k, self.s = kernel, stride
        self.circuit = QuanvCircuit(kernel * kernel, n_layers, trainable, shots)

    def forward(self, x):
        B, _, H, W = x.shape
        Ho, Wo = (H - self.k) // self.s + 1, (W - self.k) // self.s + 1
        patches = F.unfold(x, self.k, stride=self.s)                         # (B, k*k, Ho*Wo)
        patches = patches.transpose(1, 2).reshape(-1, self.k * self.k)        # (B*Ho*Wo, k*k)
        z = self.circuit(patches)                                             # (B*Ho*Wo, k*k)
        return z.view(B, Ho * Wo, -1).transpose(1, 2).reshape(B, -1, Ho, Wo)

### 4a. Validation against PennyLane (optional)
Outputs and gradients compared on random patches. Skipped if PennyLane is not installed.

In [ ]:
try:
    import pennylane as qml
    tc = QuanvCircuit(4, 2)
    dev = qml.device("default.qubit", wires=4)
    @qml.qnode(dev, interface="torch")
    def ref_circuit(p, w):
        for k in range(4): qml.RY(math.pi * p[k], wires=k)
        for l in range(w.shape[0]):
            for k in range(4): qml.Rot(w[l, k, 0], w[l, k, 1], w[l, k, 2], wires=k)
            for c in range(4): qml.CNOT(wires=[c, (c + 1) % 4])
        return [qml.expval(qml.PauliZ(k)) for k in range(4)]

    p = torch.rand(6, 4, dtype=torch.float64)
    ours = tc(p.float()).detach()
    ref = torch.stack([torch.stack(ref_circuit(pi, tc.weights.detach().double())) for pi in p]).float()
    print("max |ours - pennylane| (outputs) =", (ours - ref).abs().max().item())

    w = tc.weights.detach().double().clone().requires_grad_(True)
    torch.stack(ref_circuit(p[0], w)).sum().backward()
    tc.zero_grad(); tc(p[:1].float()).sum().backward()
    print("max |grad diff| (weights)       =", (tc.weights.grad - w.grad.float()).abs().max().item())
except ImportError:
    print("PennyLane not installed - validation skipped.")

## 5. Models

In [ ]:
class LeNetBackend(nn.Module):
    """Classical back-end shared by all front-ends: in_ch x 14 x 14 -> 10 logits."""
    def __init__(self, in_ch=4, num_classes=10):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, 16, 5)          # 14 -> 10
        self.fc1, self.fc2, self.fc3 = nn.Linear(16 * 5 * 5, 120), nn.Linear(120, 84), nn.Linear(84, num_classes)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv(x)), 2)   # 10 -> 5
        x = F.relu(self.fc1(x.flatten(1)))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class HybridQCNN(nn.Module):
    """Quanvolution (quantum) front-end + LeNet (classical) back-end."""
    def __init__(self, q_layers=2, q_trainable=True, shots=None, num_classes=10):
        super().__init__()
        self.quanv = QuanvLayer(q_layers, q_trainable, shots)
        self.backend = LeNetBackend(4, num_classes)
    def features(self, x):
        return self.quanv(x)
    def forward(self, x):
        return self.backend(self.quanv(x))


class ClassicalControl(nn.Module):
    """Same shape/range as the quanv front-end: Conv 1->4, 2x2, stride 2, tanh."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.front = nn.Conv2d(1, 4, 2, stride=2)
        self.backend = LeNetBackend(4, num_classes)
    def features(self, x):
        return torch.tanh(self.front(x))
    def forward(self, x):
        return self.backend(self.features(x))


class LeNet5(nn.Module):
    """Standard classical LeNet-5 (reference)."""
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.backend = LeNetBackend(6, num_classes)
    def forward(self, x):
        return self.backend(F.max_pool2d(F.relu(self.conv1(x)), 2))


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

model = HybridQCNN(CFG["q_layers"], CFG["q_trainable"], CFG["shots"]).to(device)
print(model)
n_q = count_params(model.quanv)
print(f"\nQuantum params: {n_q} | Classical params: {count_params(model) - n_q:,} | Total trainable: {count_params(model):,}")
print(f"Circuits per image: {14*14} (2x2 patches)  |  per batch of {CFG['batch_size']}: {14*14*CFG['batch_size']:,}")

### 5a. Print the quanvolution circuit
Module summary, weights, and the circuit drawn with PennyLane for one real 2×2 patch. Re-run after training to see the trained weights.

In [ ]:
# !pip install pennylane
import pennylane as qml

if "model" not in globals() or not isinstance(model, HybridQCNN):
    model = HybridQCNN(CFG["q_layers"], CFG["q_trainable"], CFG["shots"]).to(device)
    if os.path.exists(CFG["ckpt"]):
        model.load_state_dict(torch.load(CFG["ckpt"], map_location=device)); print(f"Loaded {CFG['ckpt']}\n")

circ = model.quanv.circuit
n, L = circ.n, circ.L
print(model.quanv)
print(f"\nRot weights {tuple(circ.weights.shape)} = {circ.weights.numel()} params | kernel 2x2, stride 2 | Hilbert dim {2**n}\n")
for l in range(L):
    print(f"Layer {l}   rows = qubits (patch pixels), cols = (phi, theta, omega)")
    print(np.round(circ.weights[l].detach().cpu().numpy(), 3))

dev = qml.device("default.qubit", wires=n)
@qml.qnode(dev)
def quanv_circuit(p, w):
    for k in range(n): qml.RY(np.pi * p[k], wires=k)                     # pixel encoding
    for l in range(L):
        for k in range(n): qml.Rot(*w[l, k], wires=k)                    # trainable
        for c in range(n): qml.CNOT(wires=[c, (c + 1) % n])              # CNOT ring
    return [qml.expval(qml.PauliZ(k)) for k in range(n)]

# a real patch: the brightest 2x2 patch of test image 0
img, lbl = test_ds[0]
pt = F.unfold(img.unsqueeze(0), 2, stride=2)[0].T                         # (196, 4)
idx = pt.sum(1).argmax().item()
p = pt[idx].numpy()
w = circ.weights.detach().cpu().numpy()
print(f"\nPatch #{idx} of test digit (label {lbl}): pixels {np.round(p, 3)}\n")
print(qml.draw(quanv_circuit, decimals=2, max_length=200)(p, w))

fig, ax = qml.draw_mpl(quanv_circuit, decimals=2, style="pennylane")(p, w)
fig.suptitle(f"Quanvolution kernel: {n} qubits (2×2 patch) × {L} layers")
fig.savefig("quanv_circuit.png", dpi=150, bbox_inches="tight"); plt.show()

with torch.no_grad():
    ours = circ(torch.tensor(p, device=device).unsqueeze(0))[0].cpu().numpy()
print("\n<Z> model     :", np.round(ours, 4))
print("<Z> PennyLane :", np.round(np.array(quanv_circuit(p, w)), 4))

## 6. Quantum feature maps before training

In [ ]:
def show_feature_maps(m, title, k=4):
    imgs = torch.stack([test_ds[i][0] for i in range(k)]).to(device)
    m.eval()
    with torch.no_grad():
        fm = m.features(imgs).cpu()
    fig, ax = plt.subplots(k, 5, figsize=(10, 2.1 * k))
    for r in range(k):
        ax[r, 0].imshow(imgs[r, 0].cpu(), cmap="gray"); ax[r, 0].set_title(f"input ({test_ds[r][1]})", fontsize=9)
        for c in range(4):
            ax[r, c + 1].imshow(fm[r, c], cmap="RdBu", vmin=-1, vmax=1); ax[r, c + 1].set_title(f"ch {c} = ⟨Z{c}⟩", fontsize=9)
    for a in ax.flat: a.axis("off")
    plt.suptitle(title); plt.tight_layout(); plt.show()

show_feature_maps(model, "Quanvolution output channels (untrained circuit)")

## 7. Training and evaluation loops

In [ ]:
def train_epoch(m, loader, opt):
    m.train(); tot, correct, n = 0.0, 0, 0
    for x, y in tqdm(loader, leave=False, desc="train"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        out = m(x); loss = F.cross_entropy(out, y)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
        tot += loss.item() * y.size(0); correct += (out.argmax(1) == y).sum().item(); n += y.size(0)
    return tot / n, 100.0 * correct / n


@torch.no_grad()
def evaluate(m, loader, return_preds=False):
    m.eval(); tot, correct, n, preds, targets = 0.0, 0, 0, [], []
    for x, y in tqdm(loader, leave=False, desc="eval"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        out = m(x); tot += F.cross_entropy(out, y).item() * y.size(0)
        p = out.argmax(1); correct += (p == y).sum().item(); n += y.size(0)
        if return_preds: preds.append(p.cpu()); targets.append(y.cpu())
    res = dict(loss=tot / n, acc=100.0 * correct / n)
    if return_preds: res["preds"], res["targets"] = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    return res


def make_optimizer(m):
    q = [p for n_, p in m.named_parameters() if "quanv" in n_ and p.requires_grad]
    ids = {id(p) for p in q}
    c = [p for p in m.parameters() if p.requires_grad and id(p) not in ids]
    groups = [{"params": c, "lr": CFG["lr"]}] + ([{"params": q, "lr": CFG["q_lr"]}] if q else [])
    return torch.optim.Adam(groups)


def fit(m, epochs, ckpt, tag):
    opt = make_optimizer(m)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist, best = dict(train_loss=[], train_acc=[], test_loss=[], test_acc=[], time=[]), 0.0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = train_epoch(m, train_dl, opt); te = evaluate(m, test_dl); sched.step()
        for k, v in zip(hist, [tl, ta, te["loss"], te["acc"], time.time() - t0]): hist[k].append(v)
        if te["acc"] > best: best = te["acc"]; torch.save(m.state_dict(), ckpt)
        print(f"[{tag}] Ep {ep:02d} | train {tl:.4f} / {ta:.2f}% | test {te['loss']:.4f} / {te['acc']:.2f}% "
              f"| best {best:.2f}% | {hist['time'][-1]:.1f}s")
    print(f"[{tag}] best test accuracy: {best:.2f}%  -> {ckpt}")
    return hist, best

## 8. Train the hybrid QCNN

In [ ]:
hist, best = fit(model, CFG["epochs"], CFG["ckpt"], "Hybrid QCNN")

## 9. Learning curves

In [ ]:
ep = np.arange(1, len(hist["train_loss"]) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, hist["train_loss"], "o-", label="train"); ax[0].plot(ep, hist["test_loss"], "s-", label="test")
ax[0].set(title="Loss", xlabel="epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, hist["train_acc"], "o-", label="train"); ax[1].plot(ep, hist["test_acc"], "s-", label="test")
ax[1].set(title="Accuracy (%)", xlabel="epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 10. Final evaluation of best checkpoint

In [ ]:
model.load_state_dict(torch.load(CFG["ckpt"], map_location=device))
res = evaluate(model, test_dl, return_preds=True)
print(f"Hybrid QCNN test accuracy: {res['acc']:.2f}%   loss: {res['loss']:.4f}\n")
print(classification_report(res["targets"], res["preds"], digits=4))

cm = confusion_matrix(res["targets"], res["preds"])
fig, ax = plt.subplots(figsize=(6.5, 5.5)); im = ax.imshow(cm, cmap="Blues")
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set(xticks=range(10), yticks=range(10), xlabel="predicted", ylabel="true", title="Confusion matrix (Hybrid QCNN)")
plt.colorbar(im, fraction=.046); plt.tight_layout(); plt.show()

## 11. Quantum feature maps after training

In [ ]:
show_feature_maps(model, "Quanvolution output channels (trained circuit)")

## 12. Robustness to finite measurement shots

In [ ]:
shot_list = [None, 4096, 1024, 256, 64, 16]
acc_S = []
for s in shot_list:
    model.quanv.circuit.shots = s
    acc_S.append(evaluate(model, test_dl)["acc"]); print(f"shots={str(s):>5}  acc={acc_S[-1]:.2f}%")
model.quanv.circuit.shots = CFG["shots"]

plt.figure(figsize=(6, 3.8))
plt.plot(range(len(shot_list)), acc_S, "o-")
plt.xticks(range(len(shot_list)), ["exact" if s is None else str(s) for s in shot_list])
plt.xlabel("shots per ⟨Z⟩ estimate"); plt.ylabel("test accuracy (%)"); plt.grid(alpha=.3)
plt.title("Accuracy vs measurement shots"); plt.tight_layout(); plt.show()

## 13. Comparison models
* **Random fixed quanv**: same circuit, weights frozen at random initialisation (Henderson et al. 2019).
* **Classical control**: Conv 1→4, 2×2, stride 2, tanh; same output shape and range as the quanv layer, so the gap isolates the quantum kernel.
* **Classical LeNet-5**: standard reference.

Set `CFG['run_baselines']=False` to skip.

In [ ]:
if CFG["run_baselines"]:
    results = {"Hybrid QCNN (trainable)": dict(hist=hist, best=best, params=count_params(model),
                                                front=f"quanv 4q×{CFG['q_layers']}L")}
    specs = [
        ("Hybrid QCNN (random fixed)", lambda: HybridQCNN(CFG["q_layers"], q_trainable=False), f"quanv 4q×{CFG['q_layers']}L, frozen"),
        ("Classical control",          lambda: ClassicalControl(),                              "conv 1→4, 2×2/s2 + tanh"),
        ("Classical LeNet-5",          lambda: LeNet5(),                                        "conv 1→6, 5×5 + pool"),
    ]
    for name, ctor, front in specs:
        set_seed(CFG["seed"])
        m = ctor().to(device)
        h, b = fit(m, CFG["epochs"], f"best_{name.split()[0].lower()}_{len(results)}.pt", name)
        results[name] = dict(hist=h, best=b, params=count_params(m), front=front)

    names = list(results)
    print(f"\n{'Model':<30}{'first layer':<28}{'params':>10}{'best acc %':>12}{'s/epoch':>10}")
    for k in names:
        r = results[k]
        print(f"{k:<30}{r['front']:<28}{r['params']:>10,}{r['best']:>12.2f}{np.mean(r['hist']['time']):>10.1f}")

    ep = np.arange(1, CFG["epochs"] + 1)
    plt.figure(figsize=(7.5, 4.3))
    for k, mk in zip(names, ["s-", "d-", "^-", "o-"]):
        plt.plot(ep, results[k]["hist"]["test_acc"], mk, label=k)
    plt.xlabel("epoch"); plt.ylabel("test accuracy (%)"); plt.grid(alpha=.3); plt.legend()
    plt.title("Test accuracy per epoch"); plt.tight_layout(); plt.show()